# 🔍 Network Scan Detector — Machine Learning for Cybersecurity

In this notebook you will build a small machine-learning model that looks at
**network connections** and decides whether each one looks like **normal traffic**
or **likely port scanning**.

Port scanning is when someone probes many ports (and often many hosts) to find
out which services are running — it is often the first step of an attack, but
it is also something IT teams do legitimately to find weaknesses.

⚠️ **All data in this notebook is synthetic (made up).** There are no real IP
addresses, hosts or networks anywhere in it.

**What we will do:**
1. Import libraries
2. Generate synthetic connection data
3. Create (imperfect) labels
4. Explore the data
5. Split into training and test sets
6. Train a decision tree
7. Evaluate it
8. Look inside the model
9. Build a `check_connection()` helper
10. Try some scenarios
11. Your turn + an evasion challenge
12. Takeaways

## 1. Imports

We only need four libraries:
- **pandas** – tables of data (DataFrames)
- **numpy** – random numbers and maths
- **matplotlib** – charts
- **scikit-learn** – the machine-learning tools

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

print("Libraries loaded ✅")

## 2. Generate synthetic connection data

We create **1000 fictional connections**. Each one has six features:

| Feature | Range | What it means |
|---|---|---|
| `ports_contacted_per_minute` | 1–500 | How many different ports this source touched per minute |
| `unique_destination_ips` | 1–100 | How many different hosts it talked to |
| `syn_only_ratio` | 0.0–1.0 | Share of *half-open* connections (SYN sent, handshake never finished) |
| `avg_packet_size_bytes` | 40–1500 | Average packet size — scan probes are tiny |
| `failed_connection_ratio` | 0.0–1.0 | Share of connections that were refused or timed out |
| `hour_of_day` | 0–23 | When the activity happened |

To make the data realistic (not just random noise), we generate two kinds of
traffic from **different distributions**:
- **Normal traffic (~80%)** – few ports, few hosts, completed handshakes,
  normal-sized packets, mostly during working hours.
- **Scan-like traffic (~20%)** – many ports, many hosts, lots of half-open
  connections, tiny packets, lots of failures, at any hour.

The two groups **overlap a little**, just like real life.

`np.random.seed(42)` makes the "random" numbers the same every time you run
the notebook, so everyone gets the same results.

In [ ]:
np.random.seed(42)   # same random numbers every run

n_total = 1000
n_scan = 200                     # about 20% scan-like
n_normal = n_total - n_scan      # about 80% normal

# ---- Normal traffic ----
normal = pd.DataFrame({
    # Poisson gives small whole numbers: most normal clients touch a few ports
    "ports_contacted_per_minute": np.random.poisson(lam=5, size=n_normal) + 1,
    "unique_destination_ips":     np.random.poisson(lam=3, size=n_normal) + 1,
    # Beta distributions give values between 0 and 1. Beta(2, 18) is usually ~0.1
    "syn_only_ratio":             np.random.beta(2, 18, size=n_normal),
    "avg_packet_size_bytes":      np.random.normal(loc=800, scale=300, size=n_normal),
    "failed_connection_ratio":    np.random.beta(2, 15, size=n_normal),
    # Normal activity mostly happens in the daytime (around 1pm)
    "hour_of_day":                np.random.normal(loc=13, scale=4, size=n_normal),
})

# ---- Scan-like traffic ----
scan = pd.DataFrame({
    "ports_contacted_per_minute": np.random.normal(loc=200, scale=110, size=n_scan),
    "unique_destination_ips":     np.random.normal(loc=30, scale=20, size=n_scan),
    # Beta(8, 3) is usually ~0.7: lots of half-open connections
    "syn_only_ratio":             np.random.beta(8, 3, size=n_scan),
    "avg_packet_size_bytes":      np.random.normal(loc=90, scale=60, size=n_scan),
    "failed_connection_ratio":    np.random.beta(6, 3, size=n_scan),
    # Scans can happen at any hour
    "hour_of_day":                np.random.uniform(0, 24, size=n_scan),
})

# Stack the two groups together and shuffle the rows
df = pd.concat([normal, scan], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Keep every feature inside its allowed range and use whole numbers where needed
df["ports_contacted_per_minute"] = df["ports_contacted_per_minute"].clip(1, 500).round().astype(int)
df["unique_destination_ips"]     = df["unique_destination_ips"].clip(1, 100).round().astype(int)
df["syn_only_ratio"]             = df["syn_only_ratio"].clip(0, 1).round(2)
df["avg_packet_size_bytes"]      = df["avg_packet_size_bytes"].clip(40, 1500).round().astype(int)
df["failed_connection_ratio"]    = df["failed_connection_ratio"].clip(0, 1).round(2)
df["hour_of_day"]                = df["hour_of_day"].round().astype(int) % 24   # wrap 24 -> 0

print("Shape:", df.shape)
df.head()

## 3. Create labels with a risk-points rule

A model needs **labels** (the "right answers") to learn from. Here we invent a
simple rule that gives each connection **risk points**:

| Condition | Points |
|---|---|
| More than 50 ports per minute | +2 |
| 21–50 ports per minute | +1 |
| More than 20 destination IPs | +1 |
| SYN-only ratio above 0.5 | +2 |
| Failed connection ratio above 0.5 | +1 |
| Average packet size below 150 bytes | +1 |
| Between midnight and 5am | +1 |

If a connection scores **4 or more points**, we label it **1 = scan**,
otherwise **0 = normal**.

Then we **flip about 5% of the labels at random**. This simulates imperfect
ground truth — in real life, analysts make mistakes and some incidents are
mislabelled or never noticed.

In [ ]:
def risk_points(row):
    """Add up risk points for one connection using our made-up rule."""
    points = 0
    if row["ports_contacted_per_minute"] > 50:
        points += 2
    elif row["ports_contacted_per_minute"] > 20:
        points += 1
    if row["unique_destination_ips"] > 20:
        points += 1
    if row["syn_only_ratio"] > 0.5:
        points += 2
    if row["failed_connection_ratio"] > 0.5:
        points += 1
    if row["avg_packet_size_bytes"] < 150:
        points += 1
    if row["hour_of_day"] <= 5:
        points += 1
    return points

# Apply the rule to every row
df["risk_points"] = df.apply(risk_points, axis=1)
df["is_scan"] = (df["risk_points"] >= 4).astype(int)

# Flip ~5% of labels to simulate noisy, imperfect ground truth
n_flip = int(0.05 * len(df))
flip_rows = np.random.choice(df.index, size=n_flip, replace=False)
df.loc[flip_rows, "is_scan"] = 1 - df.loc[flip_rows, "is_scan"]

print(f"Flipped {n_flip} labels to simulate mistakes in the ground truth")
df.head()

> 📝 **Important note about these labels**
>
> In a real project, labels would come from **verified incidents** — connections
> that a security analyst investigated and confirmed as scans (or confirmed as
> harmless).
>
> Here we created labels **with a rule**. That means the model will mostly just
> **learn our rule back**. It cannot discover anything the rule did not already
> know. That's fine for learning how the ML workflow works, but remember it
> when you look at the (very good) results later!

## 4. Explore the data

Before training, always look at your data. First: **how many of each class do
we have?** If one class is rare, a model can get high accuracy just by always
guessing the common class.

In [ ]:
counts = df["is_scan"].value_counts().sort_index()
print("Class counts (0 = normal, 1 = scan):")
print(counts)
print()
print("Class percentages:")
print((counts / len(df) * 100).round(1))

counts.plot(kind="bar", color=["tab:blue", "tab:red"])
plt.xticks([0, 1], ["Normal (0)", "Scan (1)"], rotation=0)
plt.ylabel("Number of connections")
plt.title("Class balance")
plt.show()

Now let's compare the **average value of each feature** for normal and scan
connections. Big differences suggest a feature will be useful to the model.

In [ ]:
feature_cols = [
    "ports_contacted_per_minute",
    "unique_destination_ips",
    "syn_only_ratio",
    "avg_packet_size_bytes",
    "failed_connection_ratio",
    "hour_of_day",
]

# Group by label and take the mean of each feature, then flip it so it's easier to read
means = df.groupby("is_scan")[feature_cols].mean().T
means.columns = ["Normal (0)", "Scan (1)"]
means.round(2)

### 💬 Discussion questions
1. Which features look most different between normal and scan traffic?
2. **Could a vulnerability scanner run by your own IT team be flagged?** What
   would its features look like? Is flagging it a "mistake"?
3. **Could a slow, stealthy scan evade this?** What if an attacker only probes
   5 ports per minute, finishes every handshake and works during office hours?
4. Is `hour_of_day` a fair feature? What about night-shift workers or backups
   that run at 2am?

## 5. Split features and labels, then train/test

- **X** = the features (the inputs the model looks at)
- **y** = the label (the answer we want it to predict)

We keep **20% of the data hidden** as a test set. The model never sees it during
training, so it gives us an honest check of how well the model works on new data.

`stratify=y` makes sure both sets have the same percentage of scans.

In [ ]:
X = df[feature_cols]     # features only (we do NOT include risk_points — that would be cheating!)
y = df["is_scan"]        # labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    stratify=y,          # keep class balance the same in both sets
    random_state=42,     # reproducible split
)

print("Training set:", X_train.shape[0], "rows")
print("Test set:    ", X_test.shape[0], "rows")
print("Scan % in train:", round(y_train.mean() * 100, 1))
print("Scan % in test: ", round(y_test.mean() * 100, 1))

## 6. Train a decision tree

A **decision tree** learns a series of yes/no questions such as
*"Is syn_only_ratio > 0.5?"*. It's easy to read, which makes it great for
learning and for explaining decisions to a security team.

- `max_depth=4` – at most 4 questions deep (keeps it simple and readable)
- `min_samples_leaf=10` – each final answer must be based on at least 10
  connections (stops it memorising single noisy examples)

In [ ]:
model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)

print("Model trained ✅")

## 7. Evaluate the model

We ask the model to predict the **test set** and compare to the true labels.

- **Accuracy** – the % of predictions that were correct
- **Precision** (for scans) – of the connections we flagged, how many really were scans?
- **Recall** (for scans) – of all the real scans, how many did we catch?

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.3f}\n")

print(classification_report(y_test, y_pred, target_names=["Normal", "Scan"]))

The **confusion matrix** shows exactly which kinds of mistakes the model made.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Normal", "Scan"],
    cmap="Blues",
)
plt.title("Confusion matrix (test set)")
plt.show()

# .ravel() turns the 2x2 grid into four separate numbers
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"True Negatives  (normal, correctly allowed): {tn}")
print(f"False Positives (normal, wrongly flagged):   {fp}")
print(f"False Negatives (scan, MISSED):              {fn}")
print(f"True Positives  (scan, correctly caught):    {tp}")

### 🛡️ What do the mistakes mean for a SOC?

A **SOC** (Security Operations Centre) is the team that watches alerts all day.

- **False positive** – a normal connection flagged as a scan.
  An analyst wastes time investigating it. Too many false positives cause
  **alert fatigue**: analysts start ignoring alerts, and a real attack can slip
  through. If the response is automatic blocking, a false positive might also
  **block a real user or business service**.

- **False negative** – a real scan the model called normal.
  The attacker's reconnaissance goes **unnoticed**, and they may use what they
  learned to attack later. This is often the more dangerous mistake.

There's always a trade-off: catching more scans usually means more false alarms.
Which mistake is worse depends on the organisation.

Also remember: some of the "mistakes" here are actually caused by the **5% of
labels we flipped** — the model may be right and the label wrong!

## 8. Look inside the model

One big advantage of a decision tree is that we can **draw it**. Each box shows
the question asked, how many training connections reached that box, and which
class wins there (orange = normal, blue = scan).

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    model,
    feature_names=feature_cols,
    class_names=["Normal", "Scan"],
    filled=True,        # colour boxes by class
    rounded=True,
    fontsize=9,
)
plt.title("Decision tree for scan detection")
plt.show()

**Feature importance** tells us how much each feature helped the tree make its
decisions. Higher = more useful. Compare this with the risk-points rule from
step 3 — does the model rely on the same things?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values()

importances.plot(kind="barh", color="tab:green")
plt.xlabel("Importance")
plt.title("Feature importance")
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).round(3))

👀 Notice that the tree leans **very heavily on one feature** (`syn_only_ratio`).
In our synthetic data that single feature separates the classes almost
perfectly, so the tree barely needs the others. That's convenient for the
model — but it also tells an attacker exactly which feature to disguise.
Keep this in mind for the evasion challenge in step 11!

## 9. A `check_connection()` helper

Let's wrap the model in a friendly function. Give it the six features and it
returns:
- the **prediction** (Normal or Scan)
- a **score** between 0 and 1 (the share of training examples in that tree leaf
  that were scans)
- an **illustrative response tier**:

| Score | Response |
|---|---|
| below 0.40 | ✅ Allow and monitor |
| 0.40 – 0.74 | ⚠️ Alert analyst |
| 0.75 or above | ⛔ Block and investigate |

> ⚠️ **The score is not proof.** It is the model's best guess based on patterns
> it has seen. A high score is a reason to *look closer*, not evidence that
> someone is attacking. A human should confirm before any serious action.

In [ ]:
def check_connection(ports_contacted_per_minute, unique_destination_ips,
                     syn_only_ratio, avg_packet_size_bytes,
                     failed_connection_ratio, hour_of_day):
    """Score one connection and suggest an (illustrative) response."""
    # Put the values into a one-row DataFrame with the same columns as training
    conn = pd.DataFrame([{
        "ports_contacted_per_minute": ports_contacted_per_minute,
        "unique_destination_ips": unique_destination_ips,
        "syn_only_ratio": syn_only_ratio,
        "avg_packet_size_bytes": avg_packet_size_bytes,
        "failed_connection_ratio": failed_connection_ratio,
        "hour_of_day": hour_of_day,
    }])

    prediction = model.predict(conn)[0]
    score = model.predict_proba(conn)[0][1]   # column 1 = probability of "scan"

    if score >= 0.75:
        tier = "⛔ Block and investigate"
    elif score >= 0.40:
        tier = "⚠️ Alert analyst"
    else:
        tier = "✅ Allow and monitor"

    label = "Scan" if prediction == 1 else "Normal"
    print(f"Prediction: {label} | Score: {score:.2f} | Response: {tier}")
    return label, score, tier

## 10. Try three scenarios

**Scenario A – clearly normal browsing.** An office laptop at 10am talking to a
few web servers with normal-sized packets and completed handshakes.

In [ ]:
result = check_connection(
    ports_contacted_per_minute=3,
    unique_destination_ips=4,
    syn_only_ratio=0.05,
    avg_packet_size_bytes=950,
    failed_connection_ratio=0.02,
    hour_of_day=10,
)

**Scenario B – aggressive fast scan.** Hundreds of ports per minute across many
hosts, almost all half-open, tiny packets, at 3am.

In [ ]:
result = check_connection(
    ports_contacted_per_minute=420,
    unique_destination_ips=75,
    syn_only_ratio=0.92,
    avg_packet_size_bytes=60,
    failed_connection_ratio=0.85,
    hour_of_day=3,
)

**Scenario C – ambiguous: an authorised internal vulnerability scan.** The IT
team runs a scheduled vulnerability scanner overnight. To find live services
quickly it starts with a fast SYN sweep — exactly the same technique an
attacker would use.

Is this an attack? **No** — but does the model know that?

In [ ]:
result = check_connection(
    ports_contacted_per_minute=150,
    unique_destination_ips=60,
    syn_only_ratio=0.75,
    avg_packet_size_bytes=120,
    failed_connection_ratio=0.60,
    hour_of_day=2,
)

💡 The model says **block** — but this scan is allowed! The model only sees
**behaviour**, not **intent** or **permission**. An authorised scanner behaves
a lot like an attacker's scanner. In the real world
you would handle this with an **allowlist** of approved scanner sources and
scheduled scan windows, rather than expecting the model to "just know".

## 11. ✏️ Your turn

Change the numbers below and run the cell. Try to find:
- a connection that lands in **"Alert analyst"** (the middle tier)
- the **smallest change** that moves a connection from Normal to Scan

In [ ]:
# Edit these values and re-run the cell!
result = check_connection(
    ports_contacted_per_minute=40,
    unique_destination_ips=10,
    syn_only_ratio=0.5,
    avg_packet_size_bytes=400,
    failed_connection_ratio=0.3,
    hour_of_day=14,
)

### 🕵️ Challenge: can you evade the model?

Pretend you're an attacker who wants to scan a network **without being
detected**. A common trick is a **"low-and-slow" scan**: probe only a few ports
per minute, spread out over hours or days, complete the handshake, and do it
during office hours so you blend in.

Run the cell below — then try to find other evasive settings.

In [ ]:
# A low-and-slow scan: still reconnaissance, but it looks a lot like normal traffic
result = check_connection(
    ports_contacted_per_minute=6,     # only a few ports at a time
    unique_destination_ips=3,         # one small group of hosts at a time
    syn_only_ratio=0.35,              # mixes in full connections
    avg_packet_size_bytes=400,        # pads packets to look normal
    failed_connection_ratio=0.40,     # still lots of closed ports...
    hour_of_day=14,                   # ...but during busy office hours
)

The model says **Normal** — even though this is a real scan in disguise.

**Think about it:**
- Why did the model miss it? (Hint: look at the decision tree and which
  features it relies on.)
- What extra information could catch it? (e.g. counting ports per source over
  **24 hours** instead of per minute, or noticing one source slowly touching
  *every* host in a subnet over a week.)
- Attackers adapt to defences. This is why models need regular retraining
  and why ML is never the only defence.

## 12. 🎓 Takeaways

1. **The ML workflow:** generate/collect data → label → explore → split →
   train → evaluate → interpret → use.
2. **Labels matter most.** Our labels came from a rule, so the model mostly
   learned the rule back. Real systems need labels from **verified incidents**.
3. **Accuracy is not everything.** Look at false positives (alert fatigue,
   blocked users) and false negatives (missed attacks).
4. **A score is not proof.** It's a signal to prioritise human attention.
5. **Models can be evaded.** Low-and-slow scans and other tricks can slip
   past a model trained on obvious examples.
6. **ML is one layer of defence**, working alongside:
   - 🧱 **IDS rules** (e.g. signature and threshold rules for known scan patterns)
   - 🚦 **Rate limiting** to slow down anyone probing too fast
   - ✅ **Allowlisting authorised scanners** so IT's own tools don't trigger alerts
   - 🧑‍💻 **Human review** by SOC analysts who understand context and intent

Well done — you've built your first ML-powered security detector! 🎉